<a href="https://colab.research.google.com/github/marianamachaddo/Prog26/blob/main/atividade_pratica_aula6_plotly_interativo.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Atividade Prática — Aula 6: Visualização Interativa com Plotly

Esta atividade foi construída com base nos slides da Aula 6, que apresentam a transição do gráfico estático para o gráfico como **aplicativo de exploração**, com foco em **hover**, **zoom**, **pan**, **filtros visuais**, **Plotly Express** e construção de componentes que mais tarde podem virar um dashboard. fileciteturn7file0

## Ideia central da aula
A interatividade existe para apoiar a investigação do gestor em tempo real.  
Mesmo assim, a aula reforça uma regra essencial: **interatividade não substitui clareza**. O gráfico precisa continuar limpo, bem titulado e orientado à decisão. fileciteturn7file0

## Regras da atividade
- O notebook orienta, mas **você deve construir os códigos**.
- Use **Plotly Express** sempre que possível.
- Após cada visual principal, escreva uma breve **interpretação humana**.
- Teste hover, zoom e isolamento por legenda antes de concluir.
- Pense como desenvolvedor: os gráficos de hoje poderão virar o dashboard de amanhã. fileciteturn7file0

## Dataset da atividade
Arquivo: `vendas_brasil_clean_aula6_plotly.csv`


## 1. Preparação do ambiente

Importe as bibliotecas necessárias.

**Sugestão:**
- `pandas`
- `plotly.express`
- `plotly.graph_objects` (opcional)


In [1]:
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go

## 2. Leitura e inspeção inicial da base

Leia o arquivo `vendas_brasil_clean_aula6_plotly.csv` em um DataFrame chamado `df`.

Depois:
1. exiba as primeiras linhas
2. verifique o tamanho da base
3. confira os tipos das colunas
4. confirme se `data_venda` está em formato adequado para análises temporais
5. identifique quais colunas podem alimentar:
   - comparação de categorias
   - evolução no tempo
   - relação entre variáveis
   - distribuição espacial


In [2]:
df = pd.read_csv('vendas_brasil_clean_aula6_plotly.csv')

df.head()
df.shape
df.info()

df['data_venda'] = pd.to_datetime(df['data_venda'])

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 560 entries, 0 to 559
Data columns (total 15 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   pedido_id       560 non-null    int64  
 1   data_venda      560 non-null    object 
 2   uf              560 non-null    object 
 3   canal_venda     560 non-null    object 
 4   categoria       560 non-null    object 
 5   produto         560 non-null    object 
 6   quantidade      560 non-null    int64  
 7   desconto        560 non-null    float64
 8   preco_unitario  560 non-null    float64
 9   receita         560 non-null    float64
 10  lucro           560 non-null    float64
 11  margem_lucro    560 non-null    float64
 12  latitude        560 non-null    float64
 13  longitude       560 non-null    float64
 14  mes             560 non-null    object 
dtypes: float64(7), int64(2), object(6)
memory usage: 65.8+ KB


## 3. Matriz de decisão do analista

Os slides apresentam uma regra prática: usar interativo (Plotly) quando o painel será visto em tela/web, quando o gestor precisará responder subperguntas na hora e quando o gráfico poderá ser empacotado em um dashboard. fileciteturn7file0

### Tarefa
Responda em markdown:
1. Por que esta atividade faz mais sentido em Plotly do que em Matplotlib?
2. Em que situação você ainda preferiria um gráfico estático?
3. O destino desta análise parece mais “PDF” ou mais “tela/web”?


O uso do Plotly é mais adequado pois permite interação com os dados através de hover, zoom e filtros, facilitando a exploração em tempo real. Gráficos estáticos seriam mais indicados para relatórios em PDF. Neste caso, a análise tem como destino a visualização em tela/web, com potencial de evolução para dashboard.

## 4. Missão 1 — Barras: Receita por Canal

A missão prática da aula pede construir um gráfico de barras para responder:
**qual canal vende mais?**  
Os slides também sugerem usar o hover para injetar métricas secundárias sem poluir a tela. fileciteturn7file0

### Tarefa
1. Agregue a `receita` por `canal_venda`
2. Inclua também uma métrica secundária, como `quantidade`
3. Construa um gráfico de barras com Plotly Express
4. Ordene do maior para o menor
5. Faça o hover mostrar mais do que apenas a receita

### Perguntas
- Qual canal lidera a receita?
- O canal líder também lidera em quantidade?
- O hover ajudou a enriquecer a leitura sem poluir a tela?


In [19]:
df_bar = df.groupby('canal_venda').agg({
    'receita': 'sum',
    'quantidade': 'sum'
}).reset_index().sort_values(by='receita', ascending=False)

fig_receita_canal = px.bar(
    df_bar,
    x='canal_venda',
    y='receita',
    text='receita',
    hover_data={'quantidade': True},
    title='Receita Total por Canal de Venda',
    template='plotly_white',
    color='canal_venda'
)

fig_receita_canal.update_layout(
    title_x=0.5,
    xaxis_title='Canal de Venda',
    yaxis_title='Receita (R$)',
    showlegend=False
)

fig_receita_canal.show()

### Insight obrigatório
Escreva 2 ou 3 linhas explicando:
- quem lidera
- quem fica atrás
- o que um gestor poderia investigar a seguir


📌 Respostas
* Canal líder: Online
* Também lidera em quantidade: Sim
* Hover: Sim, enriquece a análise

📌 Insight

O canal Online lidera em receita e volume, indicando forte presença digital. O App aparece como segundo canal mais relevante, enquanto a Loja Física apresenta menor desempenho. O gestor pode investigar o ticket médio por canal.

## 5. Missão 2 — Linhas: Sazonalidade da Receita Mensal

Os slides destacam que o gráfico de linhas, com zoom e pan, é ideal para navegar no tempo e isolar picos. Também reforçam que o eixo temporal precisa estar corretamente tipado no Pandas. fileciteturn7file0

### Tarefa
1. Converta `data_venda` em data, se necessário
2. Agregue a `receita` por mês
3. Construa um gráfico de linha interativo
4. Teste o zoom nos meses de pico
5. Observe se novembro e dezembro se destacam

### Perguntas
- Existe sazonalidade?
- Quais meses chamam atenção?
- O zoom ajudou a explorar melhor a parte final da série?


In [20]:
df['mes'] = df['data_venda'].dt.to_period('M').astype(str)

df_line = df.groupby('mes')['receita'].sum().reset_index()

fig_receita_mensal = px.line(
    df_line,
    x='mes',
    y='receita',
    markers=True,
    title='Evolução Mensal da Receita',
    template='plotly_white'
)

fig_receita_mensal.update_layout(
    title_x=0.5,
    xaxis_title='Mês',
    yaxis_title='Receita (R$)'
)

fig_receita_mensal.show()

### Insight obrigatório
Explique:
- qual tendência aparece
- se há picos claros
- qual hipótese de negócio pode explicar o comportamento observado


1.  Respostas

* Existe sazonalidade: Sim
* Meses de destaque: Novembro e Dezembro
* Zoom: Ajuda na análise dos picos

2.  Insight

* A receita cresce ao longo do tempo com picos no final do ano. Isso sugere influência de datas comerciais como Black Friday e Natal. O gestor pode planejar campanhas e estoques com base nesse padrão.

## 6. Missão 3 — Dispersão: Lucro vs. Receita segmentado por Categoria

Os slides mostram o uso do gráfico de dispersão para revelar correlação entre KPIs e identificar anomalias, com grande apoio do hover. fileciteturn7file0

### Tarefa
1. Construa um scatter plot com:
   - eixo X = `receita`
   - eixo Y = `lucro`
2. Use `categoria` como cor
3. Inclua no hover:
   - `produto`
   - `canal_venda`
   - `uf`
   - `margem_lucro`
4. Tente identificar algum ponto fora da curva

### Perguntas
- Existe correlação visual entre receita e lucro?
- Há anomalias?
- O hover ajuda a transformar um ponto em um caso investigável?


In [21]:
fig_lucro_vs_receita = px.scatter(
    df,
    x='receita',
    y='lucro',
    color='categoria',
    hover_data=['produto', 'canal_venda', 'uf', 'margem_lucro'],
    title='Relação entre Receita e Lucro',
    template='plotly_white'
)

fig_lucro_vs_receita.update_layout(
    title_x=0.5,
    xaxis_title='Receita (R$)',
    yaxis_title='Lucro (R$)'
)

fig_lucro_vs_receita.show()

### Insight obrigatório
Escreva 2 ou 3 linhas dizendo:
- se a correlação parece positiva
- onde surgem pontos fora do padrão
- que ação analítica poderia ser tomada depois


📌 Respostas
* Correlação: Positiva
* Anomalias: Sim
* Hover: Essencial

📌 Insight

Há relação positiva entre receita e lucro, mas existem casos com baixa margem. Isso pode indicar custos elevados ou descontos. O gestor pode investigar esses casos.

## 7. Exploração espacial — Onde está concentrada a operação?

Os slides incluem mapas geográficos como resposta para perguntas espaciais como:
**onde está concentrada nossa operação?** fileciteturn7file0

### Tarefa
Usando `latitude` e `longitude`, crie um mapa interativo que ajude a visualizar a operação.

### Sugestões
- use tamanho ou cor para uma métrica relevante (como receita)
- teste o zoom do mapa
- observe se há concentração regional

### Perguntas
- Onde a operação parece mais concentrada?
- O mapa ajudou mais do que uma simples tabela por UF?


In [22]:
fig_mapa = px.scatter_mapbox(
    df,
    lat='latitude',
    lon='longitude',
    size='receita',
    color='receita',
    hover_name='produto',
    zoom=3,
    mapbox_style='carto-positron',
    title='Distribuição Geográfica das Vendas'
)

fig_mapa.update_layout(title_x=0.5)

fig_mapa.show()

📌 Respostas
* Concentração: Sudeste e Sul
* Melhor que tabela: Sim
📌 Insight

A operação está concentrada nessas regiões, indicando maior demanda. Outras regiões podem representar oportunidades.

## 8. A tríade da interatividade

Um slide central da aula apresenta três componentes principais:
- **Tooltips (hover)**
- **Zoom & Pan**
- **Filtros visuais / isolamento via legenda** fileciteturn7file0

### Tarefa
Escolha um dos seus gráficos interativos e descreva, em markdown:
1. O que o hover acrescenta
2. Como o zoom melhora a investigação
3. Como o clique na legenda pode ajudar a isolar séries ou categorias


Tríade da Interatividade (Gráfico de Dispersão — Receita vs Lucro)

No gráfico de dispersão que relaciona receita e lucro por categoria, os três elementos da interatividade desempenham papéis fundamentais na análise:

Hover (tooltips):
O hover acrescenta informações detalhadas sobre cada ponto, como produto, canal de venda, estado e margem de lucro. Isso permite transformar um ponto isolado em um caso investigável, sem necessidade de adicionar mais elementos visuais ao gráfico.

Zoom & Pan:
O zoom possibilita focar em regiões específicas do gráfico, especialmente onde há maior concentração de dados ou possíveis anomalias. Isso melhora a investigação ao permitir uma análise mais precisa de pontos próximos ou fora do padrão, enquanto o pan facilita a navegação entre diferentes áreas.

Legenda (filtros visuais):
O clique na legenda permite ativar ou desativar categorias, ajudando a isolar grupos específicos. Isso facilita a comparação entre categorias e a identificação de padrões ou comportamentos distintos entre elas.

## 9. Plotly Express — Máximo impacto, mínimo código

A aula mostra o Plotly Express como uma API declarativa, perfeita para DataFrames do Pandas e com geração automática de HTML/JS. fileciteturn7file0

### Tarefa
Explique em markdown:
1. O que significa dizer que Plotly Express é “declarativo”?
2. Em que ele facilita a vida do analista?
3. Como isso se conecta com a ideia de futuro dashboard em Streamlit?


1. Dizer que o Plotly Express é declarativo significa que o analista apenas descreve o que deseja visualizar (por exemplo: eixo X, eixo Y, cor, tamanho), sem precisar especificar como o gráfico será construído internamente. A biblioteca cuida automaticamente da estrutura, layout e renderização do gráfico.

2. O Plotly Express reduz significativamente a quantidade de código necessária para criar visualizações completas e interativas. Com poucas linhas, é possível gerar gráficos com hover, zoom e filtros já configurados. Isso permite que o analista foque mais na interpretação dos dados e menos na implementação técnica.

3. Essa abordagem facilita a integração com ferramentas como o Streamlit, pois os gráficos já são compatíveis com ambientes web. Como o código é simples e reutilizável, ele pode ser facilmente incorporado em aplicações interativas, acelerando a construção de dashboards e produtos de dados.

## 10. Clareza continua obrigatória

A aula reforça que um gráfico interativo ruim apenas confunde o usuário de forma mais tecnológica.  
Os princípios inegociáveis continuam sendo:
- título autoexplicativo
- eixos nomeados
- unidades claras
- remoção de lixo visual fileciteturn7file0

### Tarefa
Revise um dos seus gráficos e melhore:
- título
- rótulos de eixo
- hover
- nomes das variáveis
- aparência geral

Depois escreva:
1. O que você mudou?
2. O gráfico ficou mais claro?


In [8]:
fig_receita_canal = px.bar(
    df_bar,
    x='canal_venda',
    y='receita',
    text='receita',
    hover_data={
        'receita': ':.2f',
        'quantidade': True
    },
    title='Receita Total por Canal de Venda (R$)'
)

fig_receita_canal.update_layout(
    xaxis_title='Canal de Venda',
    yaxis_title='Receita Total (R$)',
    title_x=0.5
)

fig_receita_canal.update_traces(
    texttemplate='R$ %{text:.2f}',
    textposition='outside'
)

fig_receita_canal.show()

## 11. O gráfico não fala sozinho

O último slide deixa explícito: mesmo com zoom e tooltips avançados, a interpretação humana continua insubstituível. Sempre forneça uma conclusão textual acompanhando o visual. fileciteturn7file0

### Tarefa
Escolha **dois gráficos** que você construiu e, para cada um, escreva:
- um insight principal
- uma possível decisão gerencial
- uma pergunta adicional que o gestor poderia fazer em seguida


Gráfico 1 — Receita por Canal (Barras)

Insight principal:
O canal Online lidera tanto em receita quanto em volume de vendas, indicando forte desempenho e preferência dos clientes por canais digitais.

Decisão gerencial:
Aumentar o investimento no canal Online, com foco em marketing digital, otimização da experiência do usuário e expansão de campanhas promocionais.

Pergunta adicional:
Qual é o ticket médio por canal e como ele impacta a rentabilidade de cada um?



Gráfico 2 — Evolução Mensal da Receita (Linha)

Insight principal:
A receita apresenta crescimento ao longo do tempo, com picos significativos nos meses de novembro e dezembro, indicando sazonalidade.

Decisão gerencial:
Planejar antecipadamente campanhas comerciais, estoques e logística para o final do ano, aproveitando o aumento da demanda.

Pergunta adicional:
Quais produtos ou categorias são responsáveis pelos picos de receita nesses períodos?

## 12. Missão prática final — Mapeando e construindo

Com base no quadro de missão prática da aula, entregue no notebook, no mínimo: fileciteturn7file0

1. **Barras** — Receita por canal, com hover enriquecido
2. **Linhas** — Sazonalidade da receita mensal, explorando zoom
3. **Dispersão** — Lucro vs. Receita segmentado por categoria, com hover e identificação de anomalia

### Extra recomendado
4. **Mapa** — concentração geográfica da operação

### Para cada gráfico
Inclua abaixo:
- a pergunta de negócio
- as colunas necessárias
- a interpretação humana final


In [17]:
fig_receita_canal = px.bar(
    df_bar,
    x='canal_venda',
    y='receita',
    text='receita',
    hover_data={
        'receita': ':.2f',
        'quantidade': True
    },
    title='Receita Total por Canal de Venda (R$)'
)

fig_receita_canal.update_layout(
    xaxis_title='Canal de Venda',
    yaxis_title='Receita Total (R$)',
    title_x=0.5
)

fig_receita_canal.update_traces(
    texttemplate='R$ %{text:.2f}',
    textposition='outside'
)

fig_receita_canal.show()

📌 Pergunta de negócio

Qual canal de venda gera mais receita?

📌 Colunas necessárias

canal_venda, receita, quantidade

📌 Interpretação humana

O canal Online lidera tanto em receita quanto em volume de vendas, indicando forte preferência dos clientes por esse meio. O App também apresenta desempenho relevante, enquanto a Loja Física possui menor participação. Isso sugere uma tendência de digitalização do consumo e pode orientar decisões estratégicas de investimento.

In [10]:
df['mes'] = df['data_venda'].dt.to_period('M').astype(str)

df_line = df.groupby('mes')['receita'].sum().reset_index()

fig_receita_mensal = px.line(
    df_line,
    x='mes',
    y='receita',
    markers=True,
    title='Evolução Mensal da Receita'
)

fig_receita_mensal.show()

📌 Pergunta de negócio

Como a receita evolui ao longo do tempo?

📌 Colunas necessárias

data_venda, receita

📌 Interpretação humana

A receita apresenta crescimento ao longo do tempo, com picos evidentes nos meses de novembro e dezembro. Esse comportamento indica sazonalidade, provavelmente associada a eventos comerciais como Black Friday e Natal. O gestor pode usar essa informação para planejamento estratégico de vendas e estoque.

In [11]:
fig_lucro_vs_receita = px.scatter(
    df,
    x='receita',
    y='lucro',
    color='categoria',
    hover_data=['produto', 'canal_venda', 'uf', 'margem_lucro'],
    title='Relação entre Receita e Lucro'
)

fig_lucro_vs_receita.show()

📌 Pergunta de negócio

Existe relação entre receita e lucro? Há casos fora do padrão?

📌 Colunas necessárias

receita, lucro, categoria, produto, canal_venda, uf, margem_lucro

📌 Interpretação humana

Há uma relação positiva entre receita e lucro, indicando consistência no desempenho das vendas. No entanto, alguns pontos apresentam alta receita com lucro reduzido, sugerindo margens baixas. Esses casos devem ser investigados para entender possíveis custos elevados ou estratégias de desconto.

In [12]:
fig_mapa = px.scatter_mapbox(
    df,
    lat='latitude',
    lon='longitude',
    size='receita',
    color='receita',
    hover_name='produto',
    zoom=3,
    mapbox_style='open-street-map',
    title='Distribuição Geográfica das Vendas'
)

fig_mapa.show()

📌 Pergunta de negócio

Onde está concentrada a operação de vendas?

📌 Colunas necessárias

latitude, longitude, receita, produto

📌 Interpretação humana

A operação apresenta maior concentração nas regiões Sudeste e Sul do Brasil, indicando maior volume de vendas nessas áreas. Regiões com menor presença podem representar oportunidades de expansão. O uso do mapa facilita a identificação visual desses padrões.

## 13. Mindset de desenvolvedor

Um dos slides diz que os gráficos de hoje são o dashboard de amanhã.  
Isso exige padronização desde já: nomes coerentes, lógica limpa e reaproveitamento futuro. fileciteturn7file0

### Tarefa
Responda em markdown:
1. Como você nomearia seus objetos `fig_...` de forma organizada?
2. Que partes do seu código poderiam ser reaproveitadas numa aplicação web?
3. O que vale a pena padronizar desde esta aula?


Os objetos devem seguir um padrão claro e descritivo, facilitando a leitura e manutenção do código. Um padrão recomendado é utilizar nomes baseados na métrica e no tipo de gráfico, como:

fig_receita_canal (barras)
fig_receita_mensal (linha)
fig_lucro_vs_receita (dispersão)
fig_mapa_vendas (mapa)

Esse padrão evita ambiguidades e torna o código mais compreensível, principalmente em projetos maiores.

Diversos elementos podem ser reutilizados em uma aplicação web, como:

Funções de agregação de dados (groupby e cálculos de métricas)
Estrutura dos gráficos com Plotly Express
Configurações de layout (títulos, eixos, cores)
Padrões de hover e formatação

Esses componentes podem ser facilmente integrados em aplicações com Streamlit, permitindo a criação de dashboards interativos com pouco esforço adicional.

Desde o início, é importante padronizar:

* Nomes de variáveis e objetos (seguindo um padrão consistente)
* Estrutura de criação dos gráficos
* Estilo visual (cores, templates, fontes)
* Formatação de valores (ex: moeda, casas decimais)
* Organização do código (separação por etapas: preparação, análise e visualização)

Essa padronização facilita a escalabilidade do projeto e reduz o retrabalho na construção de dashboards futuros.

## 14. Desafio extra (opcional)

Crie mais um gráfico interativo à sua escolha, desde que ele responda uma pergunta real. Exemplos:
- receita por UF (barras)
- preço unitário ao longo do tempo (linhas)
- desconto vs. quantidade (dispersão)
- receita por categoria (barras horizontais)

Mas atenção:
- use hover com intenção
- teste zoom se fizer sentido
- escreva interpretação final


In [18]:
df_categoria = df.groupby('categoria').agg({
    'receita': 'sum',
    'quantidade': 'sum'
}).reset_index().sort_values(by='receita', ascending=True)

fig_receita_categoria = px.bar(
    df_categoria,
    x='receita',
    y='categoria',
    orientation='h',
    text='receita',
    hover_data={'quantidade': True},
    title='Receita Total por Categoria de Produto',
    template='plotly_white',
    color='receita',
    color_continuous_scale='Blues'
)

fig_receita_categoria.update_traces(
    texttemplate='R$ %{text:,.0f}',
    textposition='outside'
)

fig_receita_categoria.update_layout(
    title_x=0.5,
    xaxis_title='Receita (R$)',
    yaxis_title='Categoria',
    margin=dict(t=60, b=40, l=80, r=40)
)

fig_receita_categoria.show()


1. Pergunta de negócio

Quais categorias de produtos geram mais receita?

2. Uso da interatividade
* Hover: mostra a quantidade vendida junto da receita
* Zoom: permite focar em categorias específicas
* Visual: barras horizontais facilitam leitura de nomes longos
3.  Interpretação final

As categorias com maior receita se destacam como principais impulsionadoras do faturamento da empresa. A análise mostra que nem sempre a categoria com maior receita é a que possui maior volume de vendas, o que pode indicar diferenças no preço médio ou posicionamento dos produtos. Como ação, o gestor pode avaliar a rentabilidade por categoria e definir estratégias para expandir as mais lucrativas ou melhorar o desempenho das demais.

4. Insight

Categorias com alta receita e menor volume podem indicar produtos de maior valor agregado, enquanto categorias com alto volume e menor receita podem demandar revisão de preço ou estratégia comercial.

## 15. Entrega esperada

Seu notebook deve demonstrar:
- uso correto de Plotly Express
- escolha adequada do gráfico para a pergunta
- exploração consciente de hover, zoom, pan e legenda
- compromisso com clareza
- interpretação textual consistente

### Mensagem principal da aula
A tecnologia explora.  
Mas só o analista transforma exploração em decisão. fileciteturn7file0
